In [ ]:
from dig.xgraph.method import SubgraphX
from dig.xgraph.utils.init import fix_random_seed
fix_random_seed(27)

In [27]:
from model import MolNet, XtoCtoY, XtoC, FC, SideChannel, CtoY, CBMDataset
import torch.nn as nn
from torch.utils.data import Dataset
import torch
import pandas as pd

In [29]:
# load test data from dili
test_dili_path = "data/dili_pysmash-func/test.csv"
test_dili_df = pd.read_csv(test_dili_path)

In [31]:
# utilise https://github.com/divelab/DIG/blob/dig-stable/dig/xgraph/dataset/mol_dataset.py for loading our datasets?
from dig.xgraph.dataset import MoleculeDataset

In [14]:
# full model with GNN encoder + CBM with only side channel logits as output
class GNN(nn.Module):
    def __init__(self, encoder: MolNet, cbm: XtoCtoY):
        super(GNN, self).__init__()
        self.encoder = encoder
        self.cbm = cbm

    def forward(self, x):
        embeddings = self.encoder(x)
        _, _, _, _, SC_probs = self.cbm(embeddings)
        return SC_probs

In [15]:
encoder_trained = torch.load(f'models/encoder_gnn_dili_no_1.1.pth', weights_only = False)
XtoCtoY_trained = torch.load(f'models/ModelXtoCtoY_gnn_dili_no_1.1.pth', weights_only = False)
gnn_model = GNN(encoder_trained, XtoCtoY_trained)

In [18]:
# define explainer module for side channel according to 
# https://github.com/divelab/DIG/blob/dig-stable/tutorials/KDD2022/xgraph_code_tutorial.ipynb
subgraphx_explainer = SubgraphX(gnn_model,
                                num_classes = 2,
                                device = "cpu",
                                explain_graph = False,
                                reward_method = "nc_mc_l_shapley")

In [ ]:
_, explanation_results, related_preds = subgraphx_explainer()